# DLI + Hudi CDC Demo Notebook

This notebook is the open-source Jupyter entry point for the Huawei Cloud demo. It validates the migration config, builds DLI Spark job payloads, and can call the deployment scripts from notebook cells. Cloud execution cells are manual by default.

In [ ]:
import json
import os
import platform
import subprocess
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'config' / 'job-config.json').exists():
    ROOT = ROOT.parent
assert (ROOT / 'config' / 'job-config.json').exists(), ROOT

config = json.loads((ROOT / 'config' / 'job-config.json').read_text(encoding='utf-8'))
assert len(config['tables']) == 21
print('demo root:', ROOT)
print('tables:', len(config['tables']))

In [ ]:
def env_bool(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return default
    return raw.strip().lower() in {'1', 'true', 'yes', 'y'}

BUCKET = os.environ.get('DEMO_BUCKET', 'docktest')
ENGINE = os.environ.get('DEMO_ENGINE', 'dli')
DLI_QUEUE = os.environ.get('DLI_QUEUE_NAME', 'default')
DLI_AGENCY_NAME = os.environ.get('DLI_AGENCY_NAME', 'dli_management_agency')
MRS_CLUSTER_ID = os.environ.get('MRS_CLUSTER_ID', '')
TRANSIENT_MRS_CLUSTER = env_bool('TRANSIENT_MRS_CLUSTER', False)
SMOKE_TABLES = int(os.environ.get('SMOKE_TABLES', '1'))
FORCE_FALLBACK_AUTH = env_bool('FORCE_FALLBACK_AUTH', True)
EXECUTE = env_bool('NOTEBOOK_EXECUTE', True)

os.environ['DEMO_BUCKET'] = BUCKET
os.environ['DLI_QUEUE_NAME'] = DLI_QUEUE
os.environ['HUAWEICLOUD_REGION'] = 'la-south-2'
os.environ['OBS_ENDPOINT'] = 'https://obs.la-south-2.myhuaweicloud.com'
os.environ['DLI_ENDPOINT'] = 'https://dli.la-south-2.myhuaweicloud.com'
os.environ['DLI_SPARK_VERSION'] = '3.3.1'
os.environ['DLI_AGENCY_NAME'] = DLI_AGENCY_NAME
os.environ.setdefault('DLI_AGENCY_URN', '')

print({'engine': ENGINE, 'bucket': BUCKET, 'queue': DLI_QUEUE, 'agency_name': DLI_AGENCY_NAME, 'mrs_cluster_id': MRS_CLUSTER_ID, 'transient_mrs_cluster': TRANSIENT_MRS_CLUSTER, 'smoke_tables': SMOKE_TABLES, 'execute': EXECUTE})

In [ ]:
first = config['tables'][0]
required = ['raw_obs_path', 'bronze_hudi_path', 'silver_hudi_path', 'record_key', 'precombine_field']
assert all(k in first for k in required)
assert first['record_key'] == 'id'
assert first['precombine_field'] == '_cdc_timestamp'
first

In [ ]:
def run_checked(command, *, cwd=ROOT):
    print('>', ' '.join(str(x) for x in command))
    completed = subprocess.run(command, cwd=str(cwd), text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f'command failed with exit code {completed.returncode}')
    return completed

def build_dli_status_url(endpoint, project_id, batch_id):
    return f"{endpoint.rstrip('/')}/v2.0/{project_id}/batches/{batch_id}/state"

assert build_dli_status_url('https://dli.example.com', 'p1', 'b1').endswith('/v2.0/p1/batches/b1/state')
print('helpers OK')

## Script-Driven Smoke Test

This cell calls the same deployment script used by the command line. With `EXECUTE = True`, it creates/checks the OBS bucket, uploads assets, submits the DLI Spark bronze/silver smoke jobs, and polls them.

In [ ]:
if platform.system() != 'Windows':
    print('Windows DPAPI PowerShell path skipped on Linux cloud notebook.')
else:
    if ENGINE == 'mrs':
        cmd = ['powershell', '-ExecutionPolicy', 'Bypass', '-File', 'scripts\\19_resume_mrs_notebook_workflow.ps1', '-Bucket', BUCKET, '-SmokeTables', str(SMOKE_TABLES)]
        if MRS_CLUSTER_ID:
            cmd.extend(['-MrsClusterId', MRS_CLUSTER_ID])
        if TRANSIENT_MRS_CLUSTER:
            cmd.append('-TransientCluster')
    else:
        cmd = [
            'powershell', '-ExecutionPolicy', 'Bypass',
            '-File', 'scripts\\12_resume_existing_bucket_smoke.ps1',
            '-Bucket', BUCKET,
            '-Queue', DLI_QUEUE,
            '-AgencyName', DLI_AGENCY_NAME,
            '-SmokeTables', str(SMOKE_TABLES),
        ]
    if FORCE_FALLBACK_AUTH:
        cmd.append('-ForceFallbackAuth')
    if EXECUTE:
        cmd.append('-Execute')
    run_checked(cmd)


## Linux JupyterHub Path

On CCE/JupyterHub, do not use DPAPI. Inject `HUAWEICLOUD_ACCESS_KEY`, `HUAWEICLOUD_SECRET_KEY`, optional `HUAWEICLOUD_SECURITY_TOKEN`, and `HUAWEICLOUD_PROJECT_ID` through Kubernetes Secret or the notebook server environment, then run the Python scripts below.

In [ ]:
if platform.system() == 'Windows':
    print('Linux/JupyterHub direct Python path skipped on Windows.')
else:
    if EXECUTE:
        run_checked(['python', 'scripts/07_create_minimal_chile_resources.py', '--execute', '--skip-dli'])
        run_checked(['python', 'scripts/02_upload_assets_to_obs.py', '--execute'])
        if ENGINE == 'mrs':
            run_checked(['python', 'scripts/17_prepare_mrs_assets.py', '--execute'])
    else:
        run_checked(['python', 'scripts/07_create_minimal_chile_resources.py', '--skip-dli'])
        run_checked(['python', 'scripts/02_upload_assets_to_obs.py'])
        if ENGINE == 'mrs':
            run_checked(['python', 'scripts/17_prepare_mrs_assets.py'])

    if ENGINE == 'mrs':
        mrs_cmd = ['python', 'scripts/18_run_mrs_dataflow_workflow.py', '--limit', str(SMOKE_TABLES), '--interval-seconds', '60', '--max-polls', '60']
        if MRS_CLUSTER_ID:
            mrs_cmd.extend(['--cluster-id', MRS_CLUSTER_ID])
        if TRANSIENT_MRS_CLUSTER:
            mrs_cmd.append('--transient-cluster')
            mrs_cmd.append('--wait-transient')
        if EXECUTE:
            mrs_cmd.append('--execute')
        run_checked(mrs_cmd)
    else:
        run_checked(['python', 'scripts/03_build_dli_payloads.py'])
        submit = ['python', 'scripts/04_submit_dli_jobs.py', '--limit', str(SMOKE_TABLES)]
        poll = ['python', 'scripts/05_poll_dli_jobs.py']
        if EXECUTE:
            submit.append('--execute')
            poll.append('--execute')
        run_checked(submit)
        run_checked(poll)
